# Data Provenance Metadata in DTC-IS

This notebook is a demonstrator from the **Digital Twin Component for Ice Sheets (DTC-IS)**, an ESA-funded, EO-driven digital twin project.

DTC-IS integrates Earth Observation, in-situ data, reanalysis products, and advanced data-driven models to provide a dynamic and science-based representation of the Greenland and Antarctic ice sheets, including their ice shelves. It supports the monitoring, understanding, and prediction of ice-sheet change, delivering actionable information for research, policy, and scenario-based exploration.

[Project Website](https://dtc-ice-sheets.org/) · [Interactive Dashboard](https://dashboards.dtc-ice-sheets.org/) · [Query API](https://query.dtc-ice-sheets.org/docs)

---

The DTC Query API can provide data provenance information for every DTC-IS job.

This is a record of how the job's outputs were produced, following the [W3C PROV](https://www.w3.org/TR/prov-overview/) standard.
The provenance describes every step of the job's workflow as a dependency graph.

Depending on availability, the provenance data may include, but is not limited to, the following attributes:

### Activity Attributes

| Attribute | Vocabulary | Description |
| --- | --- | --- |
| `prov:startedAtTime` | PROV | Timestamp for the start of module execution (from the original run in case of cached results). |
| `prov:endedAtTime` | PROV | Timestamp for the end of module execution (from the original run in case of cached results). |
| `prov:hadPlan` | PROV | Identifier for the DTC-IS module template being used, e.g. data-download-template for the DDM. |
| `dtcis:command` | DTC-IS | The full command used to run the module. |
| `dtcis:dtcWrapperVersion` | DTC-IS | Current version number of the DTC Wrapper utility, which is used to transfer artifacts and metadata between Python modules. |
| `dtcis:dtcWrapperCommand` | DTC-IS | The module command wrapped by the DTC Wrapper utility. |
| `dtcis:dtcWrapperDownloadInputs` | DTC-IS | The named artifact URLs the DTC Wrapper utility was asked to make available for the command. |
| `dtcis:dtcWrapperDownloadOutputs` | DTC-IS | The named artifact paths the DTC Wrapper utility was asked to upload after command execution. |
| `dtcis:moduleName` | DTC-IS | The official DTC-IS module identifier (e.g. data_download_module). |
| `dtcis:moduleUrl` | DTC-IS | The official DTC-IS module URL, typically a GitHub Repo URL. |
| `dtcis:moduleVersion` | DTC-IS | Current version number of the DTC-IS module. |

### Entity Attributes

| Attribute | Vocabulary | Description |
| --- | --- | --- |
| `dct:identifier` | [Dublin Core Terms](https://www.dublincore.org/specifications/dublin-core/dcmi-terms/) | Unique identifier for the artifact, currently always an S3 URL. |
| `dcat:downloadURL` | [Data Catalog Vocabulary](https://www.w3.org/TR/vocab-dcat/) | The publicly accessible HTTP URL for the artifact. |

The provenance data also records which activity generated each artifact (`wasGeneratedBy`), and which artifacts
each activity used as input (`used`).


**Notebook preparations:**
To start, we need to import several libraries and pre-prepared helper functions. These are needed to interact with the DTC Query API and to illustrate the resulting data provenance information.

In [ ]:
import sys

print("⏳ Installing dependencies...")
if "google.colab" in sys.modules:
    %pip install -q ipywidgets==7.7.1 dtc-query-client
    %pip install -q jupyter_ui_poll
    # Ensure the newest version of the helpers package is installed
    %pip uninstall -y dtc_is_notebooks
    %pip install -q --upgrade --force-reinstall --no-deps --no-cache-dir \
         git+https://github.com/DTC-Ice-Sheets/dtc_is_notebooks.git

In [ ]:
import json
from datetime import datetime

import graphviz
import prov.model as pm
from dtc_query_client import (
    CovariateAnalysisRequest,
    CovariateAnalysisType,
    IceShelfExtent,
    JobsApi,
    JobStatus,
    StateAndFateApi,
    VariableItem,
)
from dtc_query_client.helpers import iterate_job
from ipywidgets import Layout, widgets

from dtc_is_notebook_helpers.authentication_widget import request_api_token, running_in_colab
from dtc_is_notebook_helpers.cell_spinner import enable_cell_spinner, set_message
from dtc_is_notebook_helpers.uc3_helper_functions import _load_dataset_yaml

# Show an animation at the bottom of each cell while it is running
if not running_in_colab():
    enable_cell_spinner()

## Enter DTC-IS API token

Please run the cell below, paste your DTC-IS API token into the box, and press 'Submit'. If you don’t have a token yet, you’ll need to generate one first using the link next to the box. A quick test query will check the token, and once it passes you can move on to the next cell! To submit a different token you will need to run the cell again.

---

In [ ]:
token_widget = request_api_token()
await token_widget.wait_until_authenticated()
client = token_widget.get_client()

The authenticated client object will now be passed to any DTC-Query-Client calls we make. When you reload this page, you will need to reauthenticate the client using your API key.

### Run a job via the DTC Query API

Most query API endpoints return a job ID.
Let's run a Data Linkages query as an example (see the [Data Linkages notebook for more information](./DTC-IS-DataLinkages.ipynb)):

In [ ]:
variable_mapping = _load_dataset_yaml()

variables = [
    ("basal_melt_anomalies_pine_island", "basal_melt_anomalies"),
    ("subglacial_lake_discharge", "lake_discharge"),
    ("basal_melt_anomalies_thwaites", "basal_melt_anomalies"),
]

variable_items = [
    VariableItem(
        name=(
            f"{variable_mapping[dataset]['label']}: "
            f"{variable_mapping[dataset]['variables'][variable]['label']} "
            f"[{variable_mapping[dataset]['variables'][variable]['units']}]"
        ),
        dataset=dataset,
        variable=variable,
        extent=IceShelfExtent(variable_mapping[dataset]["region"]),
    )
    for dataset, variable in variables
]

res = await StateAndFateApi(client).run_covariate_analysis(
    ice_shelf_id="thwaites",
    covariate_analysis_request=CovariateAnalysisRequest(
        start_time=datetime(1997, 1, 1),
        end_time=datetime(2021, 1, 1),
        variables=variable_items,
        analysis_type=CovariateAnalysisType.CORR,
    ),
)
job_id = res.job_id
print("Started job with ID:", job_id)

Wait for the job to finish...

In [ ]:
async for result in iterate_job(client, res.job_id):
    set_message(f"Job progress: {result.progress_done}/{result.progress_total}")

assert result.status == JobStatus.SUCCEEDED, "Job did not succeed. Status: " + str(result.status)
print("Job completed successfully. Outputs:")
print(json.dumps(result.outputs, indent=2))

### Fetch data provenance information

The job's output contains many files produced during processing, but doesn't provide much information about what it all means.

Using the job ID, we can query the data provenance information:

In [ ]:
prov_data = await JobsApi(client).job_data_provenance(job_id)
print(json.dumps(prov_data, indent=2))

This data can be parsed and analysed using the [Python `prov` libary](https://prov.readthedocs.io/en/latest/).

Let's start by plotting a simplified overview of the provenance relationships between processes and outputs:

In [ ]:
doc = pm.ProvDocument.deserialize(content=json.dumps(prov_data))


def sanitise(identifier: pm.QualifiedName):
    return str(hash(identifier))


dot = graphviz.Digraph("Test", graph_attr={"rankdir": "BT"})
for rec in doc.records:
    match rec:
        case pm.ProvEntity():
            url = list(rec.get_attribute("dcat:downloadURL"))[0]
            filename = url.strip().removesuffix("/").split("/")[-1]
            if "." in filename:
                filename = filename.split(".")[-1]
            else:
                filename = "folder"
            shape = {"zarr": "cylinder", "folder": "folder"}.get(filename, "note")
            color = {"zarr": "orange", "folder": "yellow"}.get(filename, "lightyellow")

            dot.node(
                sanitise(rec.identifier),
                filename,
                shape=shape,
                style="filled",
                fillcolor=color,
            )
        case pm.ProvActivity():
            dot.node(
                sanitise(rec.identifier),
                list(rec.get_attribute("prov:hadPlan"))[0],
                shape="box3d",
                style="filled",
                fillcolor="lightblue",
            )
        case pm.ProvUsage() | pm.ProvGeneration():
            dot.edge(sanitise(rec.args[0]), sanitise(rec.args[1]))
dot

This shows how data flows through the workflow:
1. A JSON file is provided as input alongside the job parameters. This is a query polygon in GeoJSON format.
2. This JSON file is used by three separate Data Download Module calls, each downloading data for one of the requested data variables. Each one produces a folder containing multiple Zarr file.
3. The Zarr files are each processed into timeseries by the Timeseries entrypoint.
4. The Preprocessor entrypoint combines the timeseries into a single unified Zarr.
5. The Covariate Analyser processes the unified Zarr to produce a summary of data linkages, output as a JSON file.

Each of the nodes and edges in this graph is a "Provenance Record":

- The modules are "Provenance Activities", representing a process that consumes and/or produces data.
- The artifacts are "Provenance Entities", representing a piece of data.
- The arrows coming out of entities are "Was Generated By" relationships, linking an entity to the activity that produced it.
- The arrows coming out of activities are "Was Used By" relationsships, linking an activity to its inputs.

Each of these records contains a list of attributes.
Some attributes are from standard vocabularies (`prov`, `dct`, and `dcat`), providing universally understood attributes.
Other attributes are from a new dictionary (`dtc-is`) with terms specific to DTC-IS.

The widget below allows browsing the full attributes of each record:

In [ ]:
activities = widgets.Accordion()
entities = widgets.Accordion()
relations = widgets.Accordion()

for rec in doc.records:
    match rec:
        case pm.ProvEntity():
            title = str(list(rec.get_attribute("dct:identifier"))[0])
            parent = entities
        case pm.ProvActivity():
            title = str(list(rec.get_attribute("prov:hadPlan"))[0])
            parent = activities
        case pm.ProvUsage() | pm.ProvGeneration():
            entity: pm.ProvEntity = doc.get_record(str(list(rec.get_attribute("prov:entity"))[0]))[0]
            activity: pm.ProvActivity = doc.get_record(str(list(rec.get_attribute("prov:activity"))[0]))[0]
            activity_name = list(activity.get_attribute("prov:hadPlan"))[0]
            entity_name = list(entity.get_attribute("dct:identifier"))[0]
            if isinstance(rec, pm.ProvUsage):
                title = f"'{activity_name}' used '{entity_name}'"
            else:
                title = f"'{activity_name}' generated '{entity_name}'"
            parent = relations
        case _:
            continue
    items = []
    items.append(
        widgets.Label(
            value="Identifier",
            style={"font_weight": "bold", "text_color": "blue"},
            layout=Layout(justify_content="flex-end", align_items="center"),
        )
    )
    items.append(
        widgets.Label(
            value=str(rec.identifier),
            layout=Layout(border="1px solid black", padding="0.5em", overflow="scroll hidden"),
        )
    )

    for key, val in rec.attributes:
        items.append(
            widgets.Label(
                value=str(key),
                style={"font_weight": "bold"},
                layout=Layout(justify_content="flex-end", align_items="center"),
            )
        )
        items.append(
            widgets.Label(
                value=str(val), layout=Layout(border="1px solid black", padding="0.5em", overflow="scroll hidden")
            )
        )
    content = widgets.GridBox(children=items, layout=Layout(grid_template_columns=r"min-content 1fr"))
    parent.children += (content,)
    parent.set_title(len(parent.children) - 1, title)

tab = widgets.Tab()
tab.children = [activities, entities, relations]
tab.set_title(0, "Activities")
tab.set_title(1, "Entities")
tab.set_title(2, "Relations")
tab